# Паль или нет? Первая модель на табличных данных

Мы — маркетплейс. Продавцы выкладывают кроссовки: Samba, Dunk, Jordan, Salomon, Balenciaga, Rick Owens, Margiela… Часть из них — паль.

**Задача:** по объявлению предсказать вероятность, что это паль.

Код писать не нужно — запускаем ячейки по очереди (Shift + Enter) и смотрим, что происходит.

**Студенческая версия.** Заполните `___` в ячейках с `# TODO` и запускайте сверху вниз.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_curve
import warnings; warnings.filterwarnings('ignore')
rng = np.random.default_rng(7)

## 0. Создаём данные
Эту ячейку не разбираем — она просто генерирует 6000 объявлений.

In [ ]:
n_sellers = 900
seller_shady = rng.random(n_sellers) < 0.12
seller_age = np.where(seller_shady, rng.exponential(60, n_sellers), rng.exponential(700, n_sellers)).astype(int) + 1
seller_reviews = np.where(seller_shady, rng.poisson(5, n_sellers), rng.negative_binomial(1, 0.008, n_sellers))
seller_rating = np.clip(np.where(seller_shady, rng.normal(4.0, 0.6, n_sellers), rng.normal(4.7, 0.25, n_sellers)), 1, 5).round(1)

n = 6000
seller = rng.integers(0, n_sellers, n)
# модель: (бренд, рыночная цена ₽, сегмент, доля объявлений, «склонность к палю»)
catalog = {
    'Nike Air Force 1':             ('Nike', 12000, 'масс', 9, 0.6),
    'Nike Dunk Low Panda':          ('Nike', 14000, 'хайп', 8, 1.2),
    'Nike Air Max 95':              ('Nike', 18000, 'масс', 4, 0.5),
    'Air Jordan 1 High':            ('Jordan', 25000, 'хайп', 6, 1.3),
    'Adidas Samba OG':              ('Adidas', 13000, 'хайп', 9, 0.9),
    'Adidas Gazelle':               ('Adidas', 11000, 'масс', 6, 0.4),
    'New Balance 550':              ('New Balance', 13000, 'хайп', 6, 0.7),
    'New Balance 9060':             ('New Balance', 17000, 'хайп', 5, 0.8),
    'New Balance 2002R':            ('New Balance', 16000, 'масс', 4, 0.3),
    'Asics Gel-1130':               ('Asics', 12000, 'хайп', 6, 0.4),
    'Asics Gel-Kayano 14':          ('Asics', 16000, 'хайп', 4, 0.5),
    'Onitsuka Tiger Mexico 66':     ('Onitsuka Tiger', 12000, 'хайп', 5, 0.8),
    'Salomon XT-6':                 ('Salomon', 22000, 'хайп', 6, 0.9),
    'Salomon Speedcross 6':         ('Salomon', 15000, 'масс', 3, 0.2),
    'Balenciaga Triple S':          ('Balenciaga', 110000, 'люкс', 3, 2.0),
    'Balenciaga Track':             ('Balenciaga', 95000, 'люкс', 2, 1.8),
    'Rick Owens Geobasket':         ('Rick Owens', 120000, 'люкс', 2, 1.9),
    'Rick Owens DRKSHDW Ramones':   ('Rick Owens', 70000, 'люкс', 3, 1.7),
    'Maison Margiela Replica GAT':  ('Maison Margiela', 60000, 'люкс', 1.0, 0.5),
    'Maison Margiela Tabi':         ('Maison Margiela', 90000, 'люкс', 2, 1.6),
}
names = list(catalog)
w_share = np.array([catalog[m][3] for m in names], float); w_share /= w_share.sum()
model_name = rng.choice(names, n, p=w_share)
brand = np.array([catalog[m][0] for m in model_name])
segment = np.array([catalog[m][2] for m in model_name])
market = (np.array([catalog[m][1] for m in model_name]) * rng.lognormal(0, 0.12, n)).round(-2)
fake_bias = np.array([catalog[m][4] for m in model_name])
delivery = rng.choice(['Москва', 'регионы', 'из-за рубежа'], n, p=[0.45, 0.4, 0.15])
pub_date = pd.Timestamp('2026-07-01') + pd.to_timedelta(rng.integers(0, 90, n), unit='D')
reg_date_seller = pd.Timestamp('2026-09-30') - pd.to_timedelta(seller_age, unit='D')
reg_date = reg_date_seller[seller]
pub_date = pd.to_datetime(np.maximum(pub_date.values, (reg_date + pd.Timedelta(days=1)).values))
age_at_pub = (pub_date - reg_date).days.values

z = -4.4 + 1.6 * seller_shady[seller] + 1.5 * (age_at_pub < 30) + 1.2 * fake_bias + 1.0 * (delivery == 'из-за рубежа') + rng.normal(0, 0.6, n)
fake = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

# наценка к рынку: оригиналы около рынка; паль — либо подозрительно дёшево, либо «лимитка» втридорога
markup = np.where(fake == 1,
                  np.where(rng.random(n) < 0.6, rng.normal(-0.3, 0.2, n), rng.normal(0.3, 0.2, n)),
                  rng.normal(-0.03, 0.15, n))
price = (market * np.clip(1 + markup, 0.15, 3)).round(-1)
photos = np.clip(np.where(fake == 1, rng.poisson(3.5, n), rng.poisson(4.5, n)), 1, 12).astype(float)
receipt = (rng.random(n) < np.where(fake == 1, 0.25, 0.5)).astype(int)
complaint = (rng.random(n) < np.where(fake == 1, 0.9, 0.02)).astype(int)

# --- текст объявления ---
real_tpl = ['{m}, оригинал, размер {s}', '{m} в отличном состоянии, есть чек', 'Продаю {m}, носил пару раз',
            '{m}, полный комплект, коробка', '{m}, размер {s}, торг уместен', '{m} новые, не подошёл размер']
fake_hint = ['{m} реплика lux 1:1, размер {s}', '{m} type, качество топ', '{m} replica top quality',
             '{m} replica, размер {s}', '{m} копия как оригинал', 'Реплика {m}, отличное качество']
desc = []
for m, f in zip(model_name, fake):
    s = rng.integers(36, 46)
    tpl = rng.choice(fake_hint) if (f and rng.random() < 0.6) else rng.choice(real_tpl)
    desc.append(tpl.format(m=m, s=s))

verified_seller = rng.random(n_sellers) < np.where(seller_shady, 0.2, 0.7)
platform_seller = rng.choice(['ios', 'android', 'web'], n_sellers, p=[0.4, 0.45, 0.15])
city_seller = rng.choice(['Москва', 'Санкт-Петербург', 'Казань', 'Екатеринбург', 'Новосибирск'], n_sellers)
meta = [json.dumps({'дата_регистрации': r.strftime('%Y-%m-%d'), 'дата_публикации': p_.strftime('%Y-%m-%d'),
                    'платформа': platform_seller[s], 'город': city_seller[s], 'верифицирован': bool(verified_seller[s])},
                   ensure_ascii=False)
        for r, p_, s in zip(reg_date, pub_date, seller)]

df = pd.DataFrame({'модель': model_name, 'бренд': brand, 'сегмент': segment, 'цена': price, 'рыночная_цена': market,
                   'рейтинг_продавца': seller_rating[seller], 'отзывов': seller_reviews[seller],
                   'фото': photos, 'есть_чек': receipt,
                   'доставка': delivery, 'описание': desc, 'метаданные': meta, 'продавец': seller,
                   'жалоба_после_покупки': complaint, 'паль': fake})
df.loc[df['отзывов'] == 0, 'рейтинг_продавца'] = np.nan
df.loc[rng.choice(n, 150, replace=False), 'фото'] = np.nan
# --- безумные выбросы: опечатки в цене ---
bad = rng.choice(n, 12, replace=False)
df.loc[bad[:6], 'цена'] = 1                                   # «цена договорная» -> 1 ₽
df.loc[bad[6:], 'цена'] = df.loc[bad[6:], 'цена'] * 1000      # лишние нули
df.head()

## 1. Смотрим на данные

In [ ]:
df.head(10)

In [ ]:
print('Всего объявлений:', len(df))
print('Доля пали:', ___  # TODO: доля пали = среднее колонки 'паль')

Паля всего ~9%. Модель «всё оригинал» угадает 91% — но не найдёт ни одной подделки. Поэтому accuracy нам не подходит.

In [ ]:
df.describe().round(1)

👀 Посмотрите на `цена`: минимум 1 ₽, а максимум — десятки миллионов. Это ошибки.

In [ ]:
df.groupby(___)['паль'].mean().sort_values(ascending=False).round(2)

Люксовые модели подделывают чаще всего.

## 2. Выкидываем безумные выбросы: правило трёх сигм
Всё, что дальше 3σ от среднего, — выброс. Попробуем сначала прямо на цене:

In [ ]:
mu, sigma = df['цена'].mean(), df['цена'].std()
print(f'среднее = {mu:,.0f} ₽,  σ = {sigma:,.0f} ₽')
print(f'границы: от {mu - 3*sigma:,.0f} до {mu + 3*sigma:,.0f} ₽')

Граница снизу — **отрицательная**, цену 1 ₽ так не поймать. А σ огромная, потому что её раздули сами выбросы.

У цен длинный хвост (масс-маркет по 12 тысяч и люкс по 120), а три сигмы любят «колокол». Поэтому сначала берём **log(цены)** — и уже на нём три сигмы.

In [ ]:
log_price = ___  # TODO: логарифм цены
mu, sigma = log_price.mean(), log_price.std()
low, high = ___, ___  # TODO: μ ± 3σ
print(f'границы: от {np.exp(low):,.0f} до {np.exp(high):,.0f} ₽')

plt.figure(figsize=(10, 3.5))
plt.hist(log_price, bins=80)
plt.axvline(low, color='red', ls='--'); plt.axvline(high, color='red', ls='--')
plt.xlabel('log(цена)'); plt.title('Красные линии — μ ± 3σ'); plt.show()

crazy = ___  # TODO: True, если log_price вне [low, high]
print('Выбросов:', crazy.sum())
df[crazy][['модель', 'цена', 'рыночная_цена']]

In [ ]:
df = ___  # TODO: оставить только строки без выбросов
print('Осталось:', len(df))

## 2.5. Достаём данные из JSON
В колонке `метаданные` лежит JSON — строка со словарём внутри. Модель так не умеет, разбираем на отдельные колонки.

In [ ]:
print(df['метаданные'].iloc[0])

In [ ]:
meta = pd.json_normalize(df['метаданные'].apply(___))   # строка -> словарь -> колонки
meta.index = df.index
df = pd.concat([df, meta], axis=1)
df[['метаданные', 'дата_регистрации', 'дата_публикации', 'платформа', 'город', 'верифицирован']].head()

Из двух дат делаем признак: **сколько дней аккаунту в момент публикации**.

In [ ]:
df['дней_продавцу'] = ___  # TODO: разница дат публикации и регистрации в днях (pd.to_datetime, .dt.days)
df['верифицирован'] = df['верифицирован'].astype(int)

age_group = pd.cut(df['дней_продавцу'], [-1, 7, 30, 180, 365, 10000],
                   labels=['до недели', 'до месяца', 'до полугода', 'до года', 'больше года'])
df.groupby(age_group, observed=True)['паль'].agg(['mean', 'count']).round(3)

С новых аккаунтов палью торгуют в разы чаще.

## 3. Признак из будущего
`жалоба_после_покупки` появляется **после** покупки. Когда объявление только выложили — жалобы ещё нет. Это утечка, выкидываем.

In [ ]:
# TODO: какую колонку нельзя использовать? Удалите её
df = df.drop(columns=[___])

## 4. Делим на train и test
На train учимся, на test проверяем. Всё, что считаем (медианы и т.п.), — только по train.

In [ ]:
train, test = train_test_split(df, test_size=0.25, stratify=___, random_state=0)
train, test = train.copy(), test.copy()
print('train:', len(train), ' test:', len(test))

## 5. Заполняем пропуски

In [ ]:
train.isna().sum()

In [ ]:
photo_median = ___  # TODO: медиана фото — по какой части данных?        # медиана — только по train!
print('Медиана фото:', photo_median)

for part in [train, test]:
    part['рейтинг_пропущен'] = ___  # TODO: 1, если рейтинг пропущен   # был ли пропуск
    part['рейтинг_продавца'] = part['рейтинг_продавца'].fillna(0)            # нет отзывов -> рейтинг 0
    part['фото'] = ___  # TODO: заполнить пропуски медианой

In [ ]:
train.groupby('рейтинг_пропущен')['паль'].mean().round(3)

У продавцов без рейтинга пали намного больше. Сам факт пропуска — полезный признак!

## 6. log(1 + x) для признаков с длинным хвостом

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(train['отзывов'], bins=50); ax[0].set_title('отзывов')
ax[1].hist(np.log1p(train['отзывов']), bins=50, color='green'); ax[1].set_title('log(1 + отзывов)')
plt.show()

In [ ]:
for part in [train, test]:
    part['log_отзывов'] = ___  # TODO: log(1 + x)
    part['log_дней'] = ___

+1 нужен, чтобы не считать log(0) у продавцов без отзывов.

## 7. Модель
Одна функция, которая: превращает категории в столбцы 0/1, масштабирует, обучает логрег и возвращает вероятности на test.

In [ ]:
def train_model(features):
    X_train = pd.get_dummies(train[features]).astype(float)
    X_test = pd.get_dummies(test[features]).astype(float).reindex(columns=X_train.columns, fill_value=0)
    scaler = StandardScaler().fit(___)                      # μ и σ — по train
    model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), train['паль'])
    p = model.predict_proba(scaler.transform(X_test))[:, ___]    # вероятность «паль»
    print('ROC-AUC на test:', round(roc_auc_score(test['паль'], p), 3))
    return model, scaler, X_train, p

In [ ]:
features = ['рейтинг_продавца', 'рейтинг_пропущен', 'log_отзывов', 'log_дней', 'верифицирован', 'фото', 'есть_чек', 'сегмент', 'доставка', 'платформа']
model, scaler, X_train, p = train_model(features)

## 8. Текст объявления

In [ ]:
test[['модель', 'описание', 'паль']].head(10)

In [ ]:
for part in [train, test]:
    text = part['описание'].str.lower()
    part['слово_replica'] = text.str.contains('replica|реплик').astype(int)
    part['слово_type'] = text.str.contains(___).astype(int)
    part['слово_копия'] = text.str.contains(___)  # TODO: «копия», «1:1».astype(int)

train.groupby('слово_replica')['паль'].mean().round(2)

Слово «replica» — почти всегда паль. Добавим слова в модель:

In [ ]:
features_text = features + [___]  # TODO: добавьте текстовые флаги
model, scaler, X_train, p = train_model(features_text)

### ⚠️ Ловушка: Maison Margiela **Replica** GAT
У Margiela есть модель, которая так и называется — **Replica**. Это оригинальные кроссовки! Что модель думает про них?

In [ ]:
gat_real = (test['модель'] == 'Maison Margiela Replica GAT') & (test['паль'] == 0)
other_real = (test['модель'] != 'Maison Margiela Replica GAT') & (test['паль'] == 0)
print('Оригинальные GAT — средняя вероятность пали:', round(p[gat_real].mean(), 2))
print('Остальные оригиналы — средняя вероятность пали:', round(p[other_real].mean(), 2))
print('Сколько оригинальных GAT модель назовёт палем (порог 0.3):', f'{(p[gat_real] > 0.3).mean():.0%}')

Модель выучила «replica → паль» и обвиняет честных продавцов GAT. **Одно и то же слово значит разное в разном контексте.**

### Чиним: ищем слово replica только вне названия модели

In [ ]:
for part in [train, test]:
    text_without_model = [___  # TODO: убрать название модели из описания for d, m in zip(part['описание'], part['модель'])]
    part['слово_replica'] = pd.Series(text_without_model, index=part.index).str.contains('replica|реплик').astype(int)

model, scaler, X_train, p = train_model(features_text)
print('Оригинальные GAT — средняя вероятность пали:', round(p[gat_real].mean(), 2))
print('Сколько оригинальных GAT назовём палем (порог 0.3):', f'{(p[gat_real] > 0.3).mean():.0%}')

## 9. Порог и матрица ошибок

In [ ]:
def show(threshold):
    pred = ___  # TODO: 1, если вероятность ≥ порога
    print(f'порог {threshold}: precision = {precision_score(test["паль"], pred):.2f}, recall = {recall_score(test["паль"], pred):.2f}')
    ConfusionMatrixDisplay(confusion_matrix(test['паль'], pred), display_labels=['оригинал', 'паль']).plot(cmap='Blues')
    plt.show()

show(0.5)
show(0.2)

- Порог ниже → ловим больше пали (recall ↑), но чаще ошибаемся на оригиналах (precision ↓).
- Какой порог выбрать — зависит от того, что для нас хуже.

## 10. Выбираем порог на кросс-валидации
Порог 0.5 — не обязательно лучший. Подберём порог, при котором **F1** максимальный.

Пали в данных мало, поэтому часто включают `class_weight='balanced'` — ошибки на редком классе штрафуются сильнее. Вероятности при этом завышаются, и порог 0.5 перестаёт быть разумным.

Подбирать порог по test нельзя — тогда test перестанет быть честной проверкой. Поэтому делаем **K-fold** на train: каждый объект train получает прогноз модели, которая его не видела.

In [ ]:
X_tr = scaler.transform(X_train)
model_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
p_cv = cross_val_predict(model_bal, X_tr, train['паль'],
                         cv=___  # TODO: KFold на 5 фолдов с перемешиванием
                        , method='predict_proba')[:, 1]

thresholds = np.arange(0.05, 0.95, 0.01)
f1s = [f1_score(train['паль'], (p_cv >= t).astype(int)) for t in thresholds]
best = ___  # TODO: порог с максимальным F1

plt.figure(figsize=(9, 4)); plt.plot(thresholds, f1s)
plt.axvline(best, color='red', ls='--', label=f'лучший порог {best:.2f}')
plt.xlabel('порог'); plt.ylabel('F1 на кросс-валидации'); plt.legend(); plt.show()

In [ ]:
model_bal.fit(___, ___)
X_te = pd.get_dummies(test[features_text]).astype(float).reindex(columns=X_train.columns, fill_value=0)
p_bal = model_bal.predict_proba(scaler.transform(X_te))[:, 1]
print('F1 на test при пороге 0.5: ', round(f1_score(test['паль'], (p_bal >= 0.5).astype(int)), 3))
print(f'F1 на test при пороге {best:.2f}:', round(f1_score(test['паль'], (p_bal >= best).astype(int)), 3))

## 11. ROC-AUC

In [ ]:
fpr, tpr, _ = roc_curve(___, ___)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc_score(test["паль"], p):.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray', label='случайное угадывание')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.show()

## 12. Что выучила модель
Положительный вес — признак повышает шанс пали, отрицательный — понижает.

In [ ]:
weights = pd.Series(model.coef_[0], index=X_train.columns).sort_values()
weights.plot.barh(figsize=(8, 10)); plt.title('Веса логистической регрессии'); plt.show()

## Вопросы для обсуждения
1. Почему нельзя было оставить признак «жалоба после покупки»?
2. Почему три сигмы на самой цене не поймали цену 1 ₽, а на log(цены) — поймали?
3. Почему медиану фото считаем только по train?
4. Какие ещё слова могут оказаться названием модели, а не признаком пали?
5. Почему лучший порог получился ниже 0.5?